In [1]:
!pip install -q gymnasium torch

In [2]:
%%writefile a3c.py
import torch, torch.nn as nn, torch.nn.functional as F, torch.multiprocessing as mp, gymnasium as gym

GAMMA, N_STEPS, LR, WORKERS, MAX_EP = 0.99, 5, 1e-3, 4, 200


class Net(nn.Module):
    def __init__(self, s, a):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(s, 128), nn.ReLU())
        self.actor, self.critic = nn.Linear(128, a), nn.Linear(128, 1)

    def forward(self, x):
        h = self.body(x)
        return self.actor(h), self.critic(h)

    def act(self, state):
        logits, v = self.forward(torch.tensor(state, dtype=torch.float32).unsqueeze(0))
        dist = torch.distributions.Categorical(F.softmax(logits, -1))
        a = dist.sample()
        return a.item(), dist.log_prob(a), v


def worker(wid, gmodel, opt, counter, q):
    env = gym.make("CartPole-v1")
    local = Net(env.observation_space.shape[0], env.action_space.n)
    local.load_state_dict(gmodel.state_dict())

    while counter.value < MAX_EP:
        s, _ = env.reset(); done = False; ep_r = 0
        logp, vals, rews = [], [], []

        while not done:
            a, lp, v = local.act(s)
            s2, r, term, trunc, _ = env.step(a)
            done = term or trunc
            logp.append(lp); vals.append(v); rews.append(r); ep_r += r; s = s2

            if len(rews) == N_STEPS or done:
                R = 0.0 if done else local.act(s)[2].item()
                returns = []
                for r_ in reversed(rews):
                    R = r_ + GAMMA * R
                    returns.insert(0, R)
                returns = torch.tensor(returns, dtype=torch.float32)
                adv = returns - torch.cat(vals).squeeze(-1)
                loss = -(torch.stack(logp) * adv.detach()).mean() + 0.5 * adv.pow(2).mean()

                opt.zero_grad(); loss.backward()
                for lp_, gp_ in zip(local.parameters(), gmodel.parameters()):
                    gp_._grad = lp_.grad
                opt.step()

                local.load_state_dict(gmodel.state_dict())
                logp, vals, rews = [], [], []

        with counter.get_lock():
            counter.value += 1
        q.put((wid, counter.value, ep_r))
    env.close()


def main():
    env = gym.make("CartPole-v1")
    gmodel = Net(env.observation_space.shape[0], env.action_space.n)
    gmodel.share_memory()
    opt = torch.optim.Adam(gmodel.parameters(), lr=LR)

    counter = mp.Value('i', 0)
    q = mp.Queue()

    procs = [mp.Process(target=worker, args=(i, gmodel, opt, counter, q)) for i in range(WORKERS)]
    for p in procs:
        p.start()

    for i in range(MAX_EP):
        wid, ep, r = q.get()
        if ep % 10 == 0:
            print(f"ep={ep} worker={wid} reward={r}")

    for p in procs:
        p.join()


if __name__ == "__main__":
    try:
        mp.set_start_method("spawn", force=True)
    except RuntimeError:
        pass
    main()

Writing a3c.py


In [3]:
!python a3c.py

ep=10 worker=0 reward=60.0
ep=20 worker=3 reward=86.0
ep=30 worker=1 reward=25.0
ep=40 worker=3 reward=19.0
ep=50 worker=1 reward=21.0
ep=60 worker=1 reward=15.0
ep=70 worker=0 reward=14.0
ep=80 worker=2 reward=19.0
ep=90 worker=2 reward=19.0
ep=100 worker=2 reward=19.0
ep=110 worker=2 reward=13.0
ep=120 worker=3 reward=19.0
ep=130 worker=0 reward=23.0
ep=140 worker=2 reward=15.0
ep=150 worker=2 reward=14.0
ep=160 worker=0 reward=11.0
ep=170 worker=2 reward=14.0
ep=180 worker=2 reward=8.0
ep=190 worker=3 reward=11.0
ep=200 worker=2 reward=11.0
